# Interplanetary Transfers

## Interplanetary Hohmann Transfers
For transfers to most of the planets we can assume the orbits are both circular and coplanar. The most energy efficient way for a spacecraft to transfer from one planet’s orbit to another is to use a Hohmann transfer approach.

Referring to the Figure below, the departure point **D** is at the periapse of the transfer ellipse, and the arrival point **A** is at the apoapse:


<center><img src="Images/hohmannInterplanetary.png" alt="Drawing" style="width: 600px;"/><center>


&nbsp;  
To perform the transfer between the two orbits we need to apply two $\Delta V$, the first in *D* and the second in *A*. The circular orbital speed of planet 1 relative to the Sun is given by: 
 $$V_1=\sqrt{\frac{\mu_{sun}}{R_1}}$$

 From the orbit equation we can derive the required velocity at point *D* to get the apoapse at point *A*:

 $$V_D=\sqrt{\frac{\mu_{sun}}{R_1}}\sqrt{\frac{2R_2}{R_1+R_2}}$$

 The required $\Delta V$ is then:

 $$ \Delta V_1= V_D-V_1=\sqrt{\frac{\mu_{sun}}{R_1}}\left [ \sqrt{\frac{2R_2}{R_1+R_2}}-1 \right ]$$

 Likewise, at arrival point we have

  $$\Delta V_2= V_2-V_A=\sqrt{\frac{\mu_{sun}}{R_2}}\left [ 1-\sqrt{\frac{2R_1}{R_1+R_2}}\right ]$$

For a mission from an inner planet to an outer planet the $\Delta V$ will both be positive. In the other case (from outer to inner), they will both be negative instead of positive, meaning that they are oriented along the opposite direction respect to the planet velocity.

The transfer time will be

$$T_{transfer} = \pi \sqrt{\frac{1}{\mu_{sun}}\left [\frac{(R_1+R_2)}{2}\right ]^3}$$


In [ ]:
from routines import *

# Get the Delta V to get into an interplanetary Hohmann transfer
depPlanet = 'Earth'  # departure planet
arrPlanet = 'Mars'   # arrival planet

r1= GetPlanetOrbitalRadius(depPlanet)  # departure planet
r2= GetPlanetOrbitalRadius(arrPlanet)  # arrival planet

deltaV1, deltaV2 = HohmannDeltaVs(GetPlanetMu('Sun'), r1, r2)
t = HohmannTransferTime(GetPlanetMu('Sun'), r1, r2)
print('∆V1           = ' + str(deltaV1) + ' km/sec')
print('∆V2           = ' + str(deltaV2) + ' km/sec')
print('Transfer time = ' + str(t/86400) + ' days')

Note that the above result doesn't take into account the effect of the gravity of the departure planet. If we'd like to consider it, an additional $\Delta V$ is needed to reach the **escape velocity**, which allows the spacecraft to get enough distance from the departing body to consider the Sun as the primary source of gravitational attraction. This distance is the radius ofe the sphere of influence, which will be defined later.

### Rendezvous Opportunities
The purpose of an interplanetary mission is for the spacecraft not only to intercept a planet’s orbit but also to rendezvous with the planet when it gets there. 

<center><img src="Images/interplanetaryPhasing.png" alt="Drawing" style="width: 750px;"/><center>

The Figure above shows two planets in circular orbits around the Sun. Case a) describes a transfer from an inner planet to an outer planet, case b) from an outer planet to an inner planet.

The key parameter to consider is the **phase angle**, the angle between the position vectors of the two planets at departure time: 

$$\phi = \theta_2 - \theta_1 \$$

$\phi$ varies over time and can be either positive or negative. For rendezvous to occur at the end of a Hohmann transfer, the location of planet 2 in its orbit at the time of the spacecraft’s departure from planet 1 must be such that planet 2 arrives at the apoapse of the transfer ellipse at the same time the spacecraft does.

Considering the mean motion of the two we have:

$$n_1=\frac{2\pi}{T_1}, n_2=\frac{2\pi}{T_2}$$



Assuming that we'd like to perform an Hohmann transfer from one planet to the other, during the time it takes the spacecraft to fly from orbit 1 to orbit 2, through an angle of $\pi$ radians, planet 2 must move around its circular orbit and end up at a point directly opposite planet 1’s position when the spacecraft departed. Since planet 2’s angular velocity is $n_2$, the angular distance traveled by the planet during the spacecraft’s trip
is $n_2 T_{transfer}$.

Follows that the initial phase angle shall be:

$$\phi_0=\pi - n_2 T_{transfer}$$

<center><img src="Images/interplanetaryPhaseAngle.png" alt="Drawing" style="width: 750px;"/><center>

Now, assuming $\phi_0$ is the needed phase angle at the intial time, how long it will take to repeat the same geometry? This time interval is called **synodic period**:

$$T_{syn}= \frac{2 \pi}{\begin{vmatrix}n_2-n_1\end{vmatrix}}$$


In [ ]:
from routines import *

# Get the synodic period and the initial phase angle
depPlanet = 'Earth'  # departure planet
arrPlanet = 'Mars'   # arrival planet

r1= GetPlanetOrbitalRadius(depPlanet)  # departure planet
r2= GetPlanetOrbitalRadius(arrPlanet)  # arrival planet
n1    = GetMeanMotionFromSma(r1, GetPlanetMu('Sun'))
n2    = GetMeanMotionFromSma(r2, GetPlanetMu('Sun'))
t_Tra = HohmannTransferTime(GetPlanetMu('Sun'), r1, r2)
t_Syn = 2*math.pi/(abs(n2-n1))
phase = math.pi - n2*t_Tra
print('Synodic Period = ' + str(t_Syn/86400) + ' days')
print('Phi_0          = ' + str(math.degrees(phase)) + ' deg')

## Method of Patched Conics

There are multiple methods, of varying complexity, to calculate interplanetary trajectories. Since the trajectories are perturbed by several gravitational sources, they are not really two-body trajectories that we’ve been studying. One approach to solve those problems is to account for all the gravitational perturbations directly in the equations of motion. This approach quickly becomes very complicated and computationally intensive, so it is not usually pursued.

Instead, the most common method for interplanetary trajectory calculations is called **patched conic**. The baseline of this method is that in any space domain the trajectory of a vehicle is determined by only one gravitational field, namely, the one that dominates over the others; then we patch together several conic sections to compute the entire trajectory of interest.

<center><img src="Images/patchedConics.png" alt="Drawing" style="width: 650px;"/><center>


## Sphere of Influence
The sun is the dominant celestial body in the solar system and we refer to it awhen designing the transfer orbit. However, near a given planet the influence of its own gravity exceeds that of the Sun. For example, at its surface the earth’s gravitational force is over 1600 times greater than the Sun’s. The inverse-square nature of the law of gravity means that the force of gravity $F_g$ drops off rapidly with distance $r$ from the center of attraction.

We define the **sphere of influence** as the volume inside of which the planet’s gravitational pull dominates the pull of the Sun. Can be demonstrated that the radius of this sphere is

$$r_{SOI}= r_p \sqrt{(\frac{m_p}{m_{sun}})^5} $$

where $r_p$ is the orbital radius of the planet under consideration

<center><img src="Images/soiPic.png" alt="Drawing" style="width: 550px;"/><center>

| Planet              | Mercury | Venus   | Earth   | Mars    | Jupiter | Saturn | Uranus | Neptune | Pluto  |
|---------------------|---------|---------|---------|---------|---------|--------|--------|---------|--------|
| Orbital Radius (km) | 57.9e6  | 108.2e6 | 149.6e6 | 227.9e6 | 779.3e6 | 1427e6 | 2871e6 | 4497e6  | 5913e6 |
| SOI Radius (km)     | 112e3   | 616e3   | 925e3   | 577e3   | 48.2e6  | 54.8e6 | 51.8e6 | 86.6e6  | 3.1e6  |

## Planetary Departure

In order to escape the gravitational pull of a planet, the spacecraft must travel an hyperbolic trajectory relative to the planet, arriving at its sphere of influence with a relative velocity $v_{\infty}$ (**hyperbolic excess velocity**) greater than zero.

<center><img src="Images/planetaryDeparture.png" alt="Drawing" style="width: 950px;"/><center>

&nbsp;      
The above Figures show the departure of a spacecraft on a mission from an inner planet to an outer planet and vice versa, respectively.

On a parabolic trajectory the spacecraft will arrive at the sphere of influence ($r = \infty$) with a relative speed of zero. In that case the spacecraft remains in the same orbit as the planet and does not embark upon a heliocentric elliptical path. To travel from a planet to another, $v_{\infty}$ has to be greater that zero. Assuming $R_1$ and $R_2$ the orbital radius of the two planets, the hyperbolic eccess velocity to travel from planet 1 to planet 2 its value is determined from:

 $$ v_{\infty}=\sqrt{\frac{\mu_{sun}}{R_1}}\left [ \sqrt{\frac{2R_2}{R_1+R_2}}-1 \right ]$$
 
**$ v_{\infty}$ only depends on the orbital radius of the two planets**. The velocity at perigee of the departing hyperbola to get $v_{\infty}$ is

 $$v_p=\sqrt{v_{\infty}^{2}+\frac{2 \mu_1}{r_p}}$$

Th eccentricity of the departing hyperpola is related to the hyperbolic eccess speed and the radius of the parking orbit $r_{park}$:

 $$e=1+\frac{r_{park} v_{\infty}^{2}}{\mu_1}$$
 
, where $\mu_1$ is the gravitational parameter of the departing planet.

Finally, the location of periapsis (where the $\Delta V$ must occur) is:

$$\beta = \arccos(\frac{1}{e})$$

, where $\beta$ is the angle between the opposite direction of the departing planet velocity and the injection point ($v_{\infty}$ shall be parallel to $V_{1}$). The $\beta$ angle can be measured both clockwise or counterclowise, depending on the orbit type (prograde or not):

<center><img src="Images/planetaryDeparture2.png" alt="Drawing" style="width: 550px;"/><center>

Once the spacraft is in the right parking orbit, there is one injection burn opportunity per orbit. According to the above Figure this opportunity lies on the night side of the Earth for flights to outer planets on prograde orbits, and on the day side of the Earth in the other case. We get the opposite conclusions for flight to inner planets.

In [1]:
from routines import *
# Get the Delta V to put a vehicle onto the hyperbolic departure trajectory from a circular orbit

depPlanet = 'Earth'  # departure planet
arrPlanet = 'Mars'   # arrival planet
rPeri = 250          # altitude of circular parking orbit around planet 1 (km)

r1    = GetPlanetOrbitalRadius(depPlanet) 
r2    = GetPlanetOrbitalRadius(arrPlanet)  
rp    = GetPlanetRadius(depPlanet)
rPark = rp + rPeri

v_inf     = math.sqrt(GetMu('Sun')/r1)*(math.sqrt((2*r2)/(r1+r2))-1)
vPeriCirc = math.sqrt(GetMu('Earth')/(rPark))
vPeriEsc  = math.sqrt(v_inf*v_inf + (2*GetMu(depPlanet))/(rPark))
ecc = 1 + (rPark*v_inf*v_inf)/GetMu(depPlanet)
beta = math.degrees(math.acos(1/ecc))

print('Hyperbolic eccess vel   = ' + str(v_inf) + ' km/sec')
print('Velocity at perigee     = ' + str(vPeriCirc) + ' km/sec')
print('Velocity needed         = ' + str(vPeriEsc) + ' km/sec')
print('∆V                      = ' + str(vPeriEsc-vPeriCirc) + ' km/sec')
print('Ecc departing orbit     = ' + str(ecc))
print('β                       = ' + str(beta) + (' deg'))

Hyperbolic eccess vel   = 2.943462578746347 km/sec
Velocity at perigee     = 7.754921345146096 km/sec
Velocity needed         = 11.355244695350478 km/sec
∆V                      = 3.6003233502043823 km/sec
Ecc departing orbit     = 1.1440662471175065
β                       = 29.064279075877582 deg


### Departing Hyperbolas

To get into a tangential heliocentric Hohmann transfer orbit at the edge of the Earth SOI, the orbital plane of the departure hyperbola has to include the center of mass and the velocity vector of the departing planet (i.e., the asymptotic velocity vector $v_{\infty}$ has to be along $V_1$), but apart from that, the departure orbit plane can be rotated about a line A–A which passes through the planet’s center of mass:

<center><img src="Images/departingHyperbola.png" alt="Drawing" style="width: 650px;"/><center>

Rotating the hyperbola in this way sweeps out a surface of revolution on which lie all possible departure hyperbolas. The periapse of the hyperbola traces out a circle which, for the specified parking orbit radius $r_{park}$ is the base of a cone with vertex at the center of the planet. For a direct ascent trajectory from the launch site, any limitation on the possible inclination is reflected on limitations on the possible departure hyperbolas.

The radius of this circle is: 

$$ r_{dep} = r_p \sin{\beta} = r_p \sin (\arccos(\frac{1}{e}))$$

It is small for hyperbolic orbits with low eccentricity and grows for orbits with higher eccentricity (so, higher energy).


## Heliocentric Transfer

Hohmann transfers may be the most favorable transfer orbits from an energetic point of view, but they are very sensitive to initial thrust errors, and also they take the longest time to get to arrival planet. So just a little more thrust would make sure that, with small thrust errors, the transfer orbit still intersects the target orbit, while transition time drastically decreases. Accordingly with the figure below, different transfer orbits can be undertaken:

<center><img src="Images/nonHohmannInterplanetary.png" alt="Drawing" style="width: 600px;"/><center>

While $v_{\infty}$ is parallel to the instantaneous velocity of the departing planet, this may be not true anymore for the non-Hohmann transfers. Each of the possible trajectories will have its own tranfer time, which is less than the Hohmann's.

In general, we need to solve the **Lambert Problem** to get the two $ \Delta Vs $ .

## Planetary Rendezvous

A spacecraft arrives at the sphere of influence of the target planet with a hyperbolic excess velocity $v_{\infty}$ relative to the planet.

In the case of a mission from an inner planet to an outer planet (left side of the Figure below), the spacecraft’s heliocentric approach velocity $V_A$ is smaller in magnitude than that of the planet, $V_2$. Therefore, it crosses the forward portion of the sphere of influence. 

For a Hohmann transfer, $V_A$ and $V_2$ are parallel, so the magnitude of the hyperbolic excess velocity is:

$$v_{\infty}= V_2 - V_A$$


If the mission is from an outer planet to an inner one (right side of the Figure below), then $V_A$  is greater than $V_2$, and the spacecraft crosses the rear portion of the sphere of influence. In this case

$$v_{\infty}= V_A - V_2$$

<center><img src="Images/planetaryArrival.png" alt="Drawing" style="width: 1000px;"/><center>

If the goal is to impact the planet (or its atmosphere), the **aiming radius** $\Delta$ of the approach hyperbola must be such that hyperbola’s periapse $r_p$  equals essentially the radius of the planet. If the intent is to go into orbit around the planet, then $\Delta$ must be chosen so that the $\Delta V$  burn at periapse will occur at the correct altitude above the planet. If there is no impact with the planet and no drop into a capture orbit
around the planet, then the spacecraft will simply continue past periapse on a flyby trajectory, exiting the sphere of influence with the same relative speed $v_{\infty}$ it entered, but with the velocity vector rotated through the turn angle $\delta$.

The following realtioships apply:

$$ e= 1 + \frac{r_p v_{\infty}^{2}}{\mu_2} $$

$$ \delta = 2\arcsin \frac{1}{e}$$

As discussed in the departing hypeperbola chapter, the approach hyperbola does not lie in a unique plane; we can rotate it about a line parallel to $V_{\infty}$ and passing through the target planet's center

<center><img src="Images/approachHyperbola.png" alt="Drawing" style="width: 600px;"/><center>

&nbsp;  
Different values of the aiming radius lead to different perigee radius and, in turns, different eccentricity